# FP-QSIM Simulator Quickstart

This notebook gives a compact overview of the **FP-QSIM** simulator project.

You will see:
- what the project provides
- how to run a tiny circuit with `CustomSimulatorManual`
- how to compare with a Qiskit reference statevector

## 1) What is in this project?

`fp_qsim` contains custom statevector simulators and helpers:
- `CustomSimulatorManual`: a hand-written simulator for `u` and `cx` basis gates
- `CustomSimulatorGeneral`: matrix-based simulator for arbitrary unitary gates
- `mocked_statevector`: helper to produce a Qiskit reference statevector

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile

from fp_qsim.simulator import CustomSimulatorManual
from fp_qsim.state_vector import mocked_statevector

## 2) Build and transpile a very small circuit

`CustomSimulatorManual` expects circuits transpiled to `u` and `cx`.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

compiled = transpile(qc, basis_gates=["u", "cx"])
compiled

## 3) Run the custom simulator

The return value is a flattened NumPy complex statevector of length $2^n$.

In [ ]:
sim = CustomSimulatorManual()
sv_custom = sim.run(compiled, shots=32)

print("shape:", sv_custom.shape)
print("rounded amplitudes:", np.round(sv_custom, 3))

## 4) Compare with Qiskit reference

Statevectors may differ by a global phase, so we align before comparison.

In [ ]:
def align_global_phase(reference: np.ndarray, candidate: np.ndarray) -> np.ndarray:
    anchor = int(np.argmax(np.abs(candidate)))
    if np.isclose(candidate[anchor], 0.0):
        return candidate
    return candidate * (reference[anchor] / candidate[anchor])


sv_ref = mocked_statevector(compiled)
sv_aligned = align_global_phase(sv_ref, sv_custom)

print("allclose:", np.allclose(sv_ref, sv_aligned, atol=1e-8))